# Import Libraries and Data

In [ ]:
import wrds
import pandas as pd


In [19]:
db = wrds.Connection()

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


## Data I: RCON Series 2

The balance sheet data comes from the **WRDS's Call Report Variables for Domestic Banks**, table **RCON Series 2** (`bank_all.wrds_call_rcon_2`). 
Writer use the report dated **31 December 2025**, giving one observation per bank. All amounts are in thousands of US dollars.

| Original code | Renamed as | Variable Description (as per WRDS) | Used for |
|---|---|---|---|
| RSSD9001 | `bank_id` | Bank ID (Unique to each bank) | Merge key |
| RSSD9999 | `report_date` | Date of the Call Report | Selecting the reporting period |
| RCON2122 | `total_loans` | Total loans and leases, unearned income | NPL ratio denominator |
| RCON2170 | `total_assets` | Total assets | ROA and capital ratio denominator, bank size |
| RCON3210 | `total_equity` | Total equity capital | Capital ratio numerator |
| RCON1403 | `nonaccrual_loans` | Total, nonaccrual | NPL ratio numerator |
| RCON1407 | `past_due_90` | Total, past due 90 or more, accruing | NPL ratio numerator |




In [23]:
balance = db.raw_sql("""
    SELECT
        bank_all.wrds_call_rcon_2.rssd9001 as bank_id,
        bank_all.wrds_call_rcon_2.rssd9999 as report_date,
        bank_all.wrds_call_rcon_2.rcon2122 as total_loans, 
        bank_all.wrds_call_rcon_2.rcon2170 as total_assets,
        bank_all.wrds_call_rcon_2.rcon3210 as total_equity,
        bank_all.wrds_call_rcon_2.rcon1403 as nonaccrual_loans,
        bank_all.wrds_call_rcon_2.rcon1407 as past_due_90
    FROM bank_all.wrds_call_rcon_2
    WHERE bank_all.wrds_call_rcon_2.rssd9999 = '2025-12-31'
""")
print("\nBalance Sheet's Data retrieved successfully from WRDS.")
balance.head(10)


Balance Sheet's Data retrieved successfully from WRDS.


,bank_id,report_date,total_loans,total_assets,total_equity,nonaccrual_loans,past_due_90
0,279,2025-12-31,306644.0,428509.0,48665,2014,1
1,37,2025-12-31,19485.0,86116.0,13639,53,135
2,242,2025-12-31,30386.0,56044.0,5282,223,355
3,4231,2025-12-31,65535.0,172793.0,23294,165,0
4,354,2025-12-31,98393.0,123554.0,12729,149,541
5,457,2025-12-31,79786.0,87633.0,8097,1919,7
6,505,2025-12-31,803219.0,928503.0,78402,659,736
7,1155,2025-12-31,430565.0,557123.0,58188,138,289
8,28732,2025-12-31,114263.0,171602.0,23029,20,0
9,1351,2025-12-31,150057.0,234655.0,33224,3390,1311


## Data II: Income statement and bank identifier data

The incomde and identifier data comes from the **WRDS's Call Report Variables for Domestic Banks** table **RIAD Series 1** (`bank_all.wrds_call_riad_1`). 

RIAD items come from the income statement (Schedule RI) of the Call Report. Writer use the report dated **31 December 2025**. Call Report income is cumulative year-to-date, so the December report gives net income for the full 2025 calendar year. Amounts are in thousands of US dollars.

In [25]:
bankdata = db.raw_sql("""
    SELECT
        bank_all.wrds_call_riad_1.rssd9001 AS bank_id,
        bank_all.wrds_call_riad_1.rssd9999 AS report_date,
        bank_all.wrds_call_riad_1.rssd9017 AS bank_name,
        bank_all.wrds_call_riad_1.rssd9200 AS state,
        bank_all.wrds_call_riad_1.riad4340 AS net_income
    FROM bank_all.wrds_call_riad_1
    WHERE bank_all.wrds_call_riad_1.rssd9999 = '2025-12-31'
""")

print("\nIncome Statement and Bank Identifier Data retrieved successfully from WRDS.")
bankdata.head(10)


Income Statement and Bank Identifier Data retrieved successfully from WRDS.


,bank_id,report_date,bank_name,state,net_income
0,37,2025-12-31,BANK OF HANCOCK COUNTY,GA,484
1,242,2025-12-31,FIRST COMMUNITY BANK XENIA-FLORA,IL,791
2,279,2025-12-31,"BROADSTREET BANK, SSB",TX,3083
3,354,2025-12-31,BISON STATE BANK,KS,245
4,457,2025-12-31,LOWRY STATE BANK,MN,1224
5,505,2025-12-31,BALLSTON SPA NATIONAL BANK,NY,5014
6,1155,2025-12-31,RVR BANK,NE,9430
7,1351,2025-12-31,BANK OF GRANDIN,MO,3677
8,1454,2025-12-31,HILLTOP NATIONAL BANK,WY,18172
9,1669,2025-12-31,"LIBERTY NATIONAL BANK IN PARIS, THE",TX,5815


# Exploratory Data Analysis (EDA)

What we aim in this section: 
1. Check the shapes of each dataset: The number of banks (row) and variables (columns) in the balance sheet and income tables.
2. Check missing values in each columns.
3. Merge the two datasets on bank ID.

In [30]:
print(f"\n{' Shapes of the Data ':=^60}")
print(f"Balance sheet data: {balance.shape[0]} banks, {balance.shape[1]} columns")
print(f"Income data:        {bankdata.shape[0]} banks, {bankdata.shape[1]} columns")

def missing_table(data):
    return pd.DataFrame({
        "non_missing": data.notna().sum(),
        "missing": data.isna().sum(),
        "missing_%": (100 * data.isna().mean()).round(2),
        "dtype": data.dtypes,
    })

print(f"\n{' Balance sheet data ':=^60}")
print(missing_table(balance))
print(f"\n{' Income data ':=^60}")
print(missing_table(bankdata))


==================== Shapes of the Data ====================
Balance sheet data: 4807 banks, 7 columns
Income data:        4807 banks, 5 columns

==================== Balance sheet data ====================
                  non_missing  missing  missing_%           dtype
bank_id                  4807        0       0.00           Int64
report_date              4807        0       0.00  string[python]
total_loans              4807        0       0.00         Float64
total_assets             4718       89       1.85         Float64
total_equity             4718       89       1.85           Int64
nonaccrual_loans         4718       89       1.85           Int64
past_due_90              4718       89       1.85           Int64

======================= Income data ========================
             non_missing  missing  missing_%           dtype
bank_id             4807        0        0.0           Int64
report_date         4807        0        0.0  string[python]
bank_name          